### Würfelglück – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021), 1. Runde, Aufgabe 4*

Barbara findet auf dem Dachboden eine Kiste mit vielen verschiedenen Spielwürfeln. Ein „normaler“ ist dabei, der sechs
Seiten hat mit den Zahlen 1 bis 6. Ein anderer hat auch sechs Seiten, aber dreimal die 1 und dreimal die 6. Und dann gibt es
noch welche mit ganz anderen Formen, zum Beispiel mit 4, 10, 12 oder gar 20 Seiten.

Zum großen Einsatz kommt die Kiste beim nächsten gemeinsamen Spieleabend mit ihren Freunden. Jeder darf sich einen Würfel
aussuchen und es wird „Mensch ärgere Dich nicht“ gespielt. Es kommt schnell eine Diskussion auf, welcher Würfel denn die
besten Chancen einräumt. Anna ist begeistert vom 20-seitigen Würfel, weil sie in einem Zug bis zu 20 Schritte machen kann.
Clemens hingegen findet es viel wichtiger, dass er oft eine 6 würfelt. Denn diese benötigt man weiterhin, um eine Figur auf
das Startfeld zu stellen.

#### Aufgabe 4

Hilf den Freunden und schreibe ein Programm, das eine Liste von Würfeln einliest und bestimmt, wie gut diese Würfel geeignet
sind, um das Spiel zu gewinnen. Vergleiche dazu jeden Würfel mit jedem anderen. Um festzustellen, welcher von zwei gegebenen
Würfeln besser ist, simuliere ausreichend viele Spiele zweier Personen, wobei beide je einen der Würfel verwenden. Natürlich
sollen in der Hälfte der Simulationen die erste Person anfangen und in der anderen die andere. Eine genaue Beschreibung der
für die Simulation gültigen Spielregeln findest du auf den BWINF-Webseiten.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/wuerfel0.txt` bis `wuerfel3.txt`)

- 1. Zeile: Anzahl n der Würfel
- je Würfel eine Zeile: Anzahl der Seiten, dann die Augenzahlen der Seiten

Es gelten die offiziellen Regeln von „Mensch ärgere Dich nicht“ (Schmidt Spiele) mit einer Änderung: Wer mehrere Steine auf der
Laufbahn hat, muss mit dem **vordersten** Stein ziehen, der gezogen werden kann. Gespielt wird zu zweit mit gegenüberliegenden Farben.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Simulation und Monte-Carlo-Verfahren:** Die Gewinnchance eines Würfels lässt sich kaum ausrechnen, aber leicht *ausprobieren*:
  Man spielt sehr viele Spiele am Computer und zählt, wie oft jeder gewinnt. Der Anteil nähert sich der wahren Wahrscheinlichkeit
  (Gesetz der großen Zahlen).
- **Zustand modellieren:** Von jedem Stein merkt man sich nur eine Zahl, nämlich wie weit er schon gelaufen ist. Damit ist das
  ganze Spielfeld beschrieben.
- **Regeln in feste Reihenfolge bringen:** Die Spielregeln sind eine Liste von „wenn möglich, dann zuerst …“. Genau in dieser Reihenfolge
  prüft man sie im Programm.
- **Sonderfall Patt (Deadlock):** Es gibt Würfel, mit denen sich nichts mehr bewegen lässt (zum Beispiel ohne 6). Wenn beide Spieler so
  festsitzen, würde das Spiel nie enden. Das muss man erkennen und festlegen, wie es gewertet wird.

**Nützliche Funktionen und Bibliotheken**
- `random.choice(liste)` wählt zufällig ein Element, `random.seed(zahl)` macht die Zufallszahlen wiederholbar.
- `list.index(x)`, `x in liste` und `sorted(liste, reverse=True)` für die Steine.
- `set(liste)` entfernt doppelte Werte (für die Seiten eines Würfels).

## Die Idee

**Das Spielfeld.** Die Laufbahn hat 40 Felder. Jeder Spieler hat vier Steine. Weil nur zwei spielen, sitzen sie sich gegenüber, ihre
Startfelder liegen 20 Felder auseinander. Für jeden Stein genügt **eine Zahl** `p`:

| `p` | Bedeutung |
|---|---|
| `-1` | Der Stein wartet auf der Reservebank. |
| `0` bis `39` | Der Stein steht auf der Laufbahn, `p` Felder hinter dem eigenen Startfeld. `0` ist das Startfeld selbst. |
| `40` bis `43` | Der Stein steht in einem der vier Zielfelder. |

Ein Spieler ist dann eine Liste mit vier Zahlen, am Anfang `[0, -1, -1, -1]`: ein Stein auf dem Startfeld, drei in Reserve.
Wer alle vier Zahlen `>= 40` hat, hat gewonnen.

**Wann steht ein Stein des Gegners auf meinem Feld?** Mein Feld `p` ist für den Gegner sein Feld `p + 20` (modulo 40, denn nach 39 geht es
bei 0 weiter). Ein Gegner-Stein mit der Zahl `q` steht also genau dann auf meinem Feld `t`, wenn `q == (t + 20) % 40` ist. So lassen sich Schlagzüge ohne
ein ganzes Spielbrett prüfen.

**Ein Zug** (Spielregeln H4 bis H6 der Aufgabe) in dieser Reihenfolge:

1. Gibt es noch Steine in Reserve und steht ein eigener Stein auf dem Startfeld, muss man das Startfeld **räumen**: Man zieht diesen Stein um die gewürfelte
   Zahl, wenn das geht.
2. Sonst, bei einer **6** und mit Steinen in Reserve: Man setzt einen neuen Stein auf das Startfeld. Steht dort ein gegnerischer Stein, wird er geschlagen.
3. Sonst zieht man den **vordersten** Stein, der gezogen werden kann.
4. Geht nichts, **verfällt** der Zug.

Ein Zug mit einem Stein geht nicht, wenn das Ziel ein eigener Stein ist oder hinter dem letzten Zielfeld (`p + Augen > 43`) liegt. Ein gegnerischer Stein auf der Laufbahn wird geschlagen
und kommt in die Reserve. In den Zielfeldern dürfen eigene Steine übersprungen werden. Nach jedem Zug ist der andere dran, **außer nach einer 6**, dann würfelt man noch einmal.

**Gewinnchance.** Zwei Würfel A und B spielen viele Spiele. In der Hälfte beginnt A, in der anderen B (nach jedem Spiel wechselt der Beginner). Der Anteil der Siege
von A ist die Schätzung der Gewinnwahrscheinlichkeit. Das macht man für alle Paare und erhält eine Tabelle.

**Patt.** Hat keiner mehr einen möglichen Zug (mit keiner Augenzahl seines Würfels), ändert sich nie wieder etwas. Wir werten so ein Spiel als **unentschieden**
und zählen es für beide als halben Sieg. Das passt zu den Tabellen der Lösungshinweise. Wie oft das vorkommt, schauen wir uns am Ende an.

## Schritt 1: Einlesen

Jeder Würfel ist eine Liste seiner Augenzahlen. `random.choice(würfel)` würfelt dann einmal, auch mit mehrfach vorkommenden Zahlen wie bei `1 1 1 6 6 6`.
Es gibt auch Würfel mit einer 0.

In [ ]:
import random


def lies_wuerfel(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = [zeile.split() for zeile in f.read().splitlines() if zeile.strip()]
    wuerfel = []
    for zeile in zeilen[1:]:
        zahlen = [int(z) for z in zeile]
        wuerfel.append(zahlen[1:])        # zahlen[0] ist die Seitenzahl
    return wuerfel


wuerfel = lies_wuerfel('beispieldaten/wuerfel0.txt')
for w in wuerfel:
    print(len(w), 'Seiten:', w)

6 Seiten: [1, 2, 3, 4, 5, 6]
6 Seiten: [1, 1, 1, 6, 6, 6]
4 Seiten: [1, 2, 3, 4]
10 Seiten: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
12 Seiten: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
20 Seiten: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## Schritt 2: Ein einzelner Stein zieht

`setze(eigene, gegner, p, augen)` versucht, den Stein mit der Zahl `p` um `augen` weiterzuziehen. Wenn es geht, trägt sie die neue Zahl ein, schlägt gegebenenfalls
einen Gegner und liefert `True`, sonst `False`.

- `ziel = p + augen`. Ist `ziel > 43` oder steht dort schon ein eigener Stein (`ziel in eigene`), geht es nicht. Bei `augen == 0` ist `ziel == p`, also ebenfalls nicht.
- Liegt das Ziel auf der Laufbahn (`ziel < 40`), steht dort vielleicht ein Gegner. Seine Zahl wäre `(ziel + 20) % 40`. Wenn ja, wird er auf `-1` gesetzt.

In [ ]:
def setze(eigene, gegner, p, augen):
    ziel = p + augen
    if ziel > 43 or ziel in eigene:
        return False
    if ziel < 40:
        q = (ziel + 20) % 40                  # so heißt dieses Feld für den Gegner
        if q in gegner:
            gegner[gegner.index(q)] = -1       # geschlagen: zurück in die Reserve
    eigene[eigene.index(p)] = ziel
    return True

## Schritt 3: Ein ganzer Zug

`ziehe(eigene, gegner, augen)` führt die Regeln in der oben genannten Reihenfolge aus und liefert `True`, wenn gezogen wurde.

- **Räumen:** `-1 in eigene` heißt „es gibt Reserve“, `0 in eigene` heißt „Startfeld belegt“. Dann zieht man den Stein von 0 aus. Klappt das nicht (zum Beispiel, weil auf dem Zielfeld ein eigener
  Stein steht), geht es weiter zu den nächsten Regeln.
- **Neuer Stein:** nur bei einer 6, mit Reserve und freiem Startfeld. Der Stein bekommt die Zahl 0, ein Gegner auf dem Startfeld (für ihn die Zahl 20) wird geschlagen.
- **Vorderster Stein:** `sorted(eigene, reverse=True)` liefert die Zahlen von der größten zur kleinsten. Wir probieren sie der Reihe nach und hören bei den Reservesteinen (`-1`) auf.

In [ ]:
def ziehe(eigene, gegner, augen):
    if -1 in eigene:
        if 0 in eigene:                                   # Startfeld räumen
            if setze(eigene, gegner, 0, augen):
                return True
        elif augen == 6:                                  # neuen Stein aufs Startfeld setzen
            eigene[eigene.index(-1)] = 0
            if 20 in gegner:
                gegner[gegner.index(20)] = -1
            return True
    for p in sorted(eigene, reverse=True):                # vordersten ziehbaren Stein suchen
        if p < 0:
            break
        if setze(eigene, gegner, p, augen):
            return True
    return False

### Zwischentest an kleinen Stellungen

Bevor wir Tausende Spiele simulieren, prüfen wir die Regeln an Stellungen, deren Ergebnis wir selbst vorhersagen können.

In [ ]:
# 1) Startfeld räumen geht vor: Mit einer 6 zieht der Startstein auf Feld 6 und schlägt dort den Gegner (für ihn Feld 26).
e = [0, -1, -1, -1]
g = [26, -1, -1, -1]
print(ziehe(e, g, 6), e, g)

# 2) Jetzt ist das Startfeld frei: Mit einer 6 kommt ein neuer Stein aufs Startfeld und schlägt den Gegner auf seinem Feld 20.
g = [20, 30, -1, -1]
print(ziehe(e, g, 6), e, g)

# 3) Mit einer 3 muss wieder der Stein auf dem Startfeld ziehen (0 -> 3), nicht der vorderste (6).
print(ziehe(e, g, 3), e, g)

# 4) In den Zielfeldern: 39 + 5 ist zu weit, 43 kann nicht weiter -> der Zug verfällt.
e = [39, 43, -1, -1]
print(ziehe(e, [0, -1, -1, -1], 5), e)

# 5) Eigene Steine dürfen übersprungen werden: 42 kann nicht (45), also zieht 40 auf 43 (über die 42).
e = [40, 42, -1, -1]
print(ziehe(e, [0, -1, -1, -1], 3), e)

# 6) Eine 0 bewegt nichts.
e = [5, -1, -1, -1]
print(ziehe(e, [0, -1, -1, -1], 0), e)

True [6, -1, -1, -1] [-1, -1, -1, -1]
True [6, 0, -1, -1] [-1, 30, -1, -1]
True [6, 3, -1, -1] [-1, 30, -1, -1]
False [39, 43, -1, -1]
True [43, 42, -1, -1]
False [5, -1, -1, -1]


## Schritt 4: Ein ganzes Spiel

`spiel(wuerfel_a, wuerfel_b, a_beginnt)` liefert `0`, wenn A gewinnt, `1`, wenn B gewinnt, und `-1` bei einem Patt.

- `pos[0]` und `pos[1]` sind die Steine der beiden Spieler, `am_zug` ist 0 oder 1.
- In jeder Runde wird gewürfelt und gezogen. Hat der Spieler danach alle vier Steine in den Zielfeldern (`min(...) >= 40`), ist das Spiel aus.
- Nach einer Augenzahl **ungleich 6** wechselt der Spieler (`am_zug = 1 - am_zug`).
- **Patt erkennen:** Wir zählen die Zugverfälle hintereinander (`ohne_zug`). Sind es mindestens zwei, prüft `kann_ziehen` genau, ob irgendein Spieler mit irgendeiner Seite seines
  Würfels noch ziehen könnte (an einer Kopie, damit die echte Stellung unverändert bleibt). Wenn keiner kann, bleibt die Stellung für immer so: Patt.

In [ ]:
def kann_ziehen(eigene, gegner, wuerfel):
    for augen in set(wuerfel):                 # jede Seite nur einmal probieren
        if ziehe(eigene[:], gegner[:], augen):  # [:] = Kopie
            return True
    return False


def spiel(wuerfel_a, wuerfel_b, a_beginnt):
    wuerfel = [wuerfel_a, wuerfel_b]
    pos = [[0, -1, -1, -1], [0, -1, -1, -1]]
    am_zug = 0 if a_beginnt else 1
    ohne_zug = 0
    while True:
        augen = random.choice(wuerfel[am_zug])
        if ziehe(pos[am_zug], pos[1 - am_zug], augen):
            ohne_zug = 0
            if min(pos[am_zug]) >= 40:
                return am_zug
        else:
            ohne_zug = ohne_zug + 1
            if ohne_zug >= 2:
                if not kann_ziehen(pos[0], pos[1], wuerfel_a) and not kann_ziehen(pos[1], pos[0], wuerfel_b):
                    return -1
        if augen != 6:
            am_zug = 1 - am_zug


random.seed(1)
normal = [1, 2, 3, 4, 5, 6]
print([spiel(normal, normal, k % 2 == 0) for k in range(20)])
print('Würfel 1-4 gegen 1-4:', [spiel([1, 2, 3, 4], [1, 2, 3, 4], True) for k in range(3)])

[0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1]
Würfel 1-4 gegen 1-4: [-1, -1, -1]


Zwei normale Würfel liefern mal A, mal B als Sieger. Mit zwei Würfeln `1 2 3 4` bekommt keiner je eine 6, es kommt also nie ein zweiter Stein ins Spiel: Der erste
Stein läuft ins Ziel (und kann dort nicht weiter), danach bewegt sich nichts mehr, beide stecken fest. Das Programm erkennt das Patt (`-1`) und bleibt nicht hängen.

## Schritt 5: Viele Spiele für ein Würfelpaar

`vergleiche(wuerfel_a, wuerfel_b, n)` spielt `n` Spiele, abwechselnd beginnt A und B (`k % 2`). Zurück kommen die Anzahl der Siege von A und die der Patts.
Ein Patt zählt als halber Sieg: `anteil = (siege + 0.5 * patts) / n`.

In [ ]:
def vergleiche(wuerfel_a, wuerfel_b, n):
    siege = 0
    patts = 0
    for k in range(n):
        ergebnis = spiel(wuerfel_a, wuerfel_b, k % 2 == 0)
        if ergebnis == 0:
            siege = siege + 1
        elif ergebnis == -1:
            patts = patts + 1
    return siege, patts


random.seed(2021)
n = 4000
siege, patts = vergleiche([1, 2, 3, 4, 5, 6], [1, 1, 1, 6, 6, 6], n)
print('normal gegen 1 1 1 6 6 6:', siege, 'Siege von', n, ', Patts:', patts)
print('Anteil:', round(100 * (siege + 0.5 * patts) / n, 1), 'Prozent')

normal gegen 1 1 1 6 6 6: 70 Siege von 4000 , Patts: 0
Anteil: 1.8 Prozent


Der Würfel `1 1 1 6 6 6` ist dem normalen Würfel haushoch überlegen: Der normale gewinnt nur etwa 2 von 100 Spielen. Der Grund liegt vermutlich darin, dass er in der Hälfte der Würfe
eine 6 würfelt (statt in einem Sechstel), also schneller neue Steine ins Spiel bringt und nach jeder 6 nochmal würfeln darf. Die großen Schritte (bis 6) braucht er kaum.

## Schritt 6: Die ganze Tabelle

Für alle Paare `i < j` simulieren wir `n` Spiele. Für das umgekehrte Paar gilt `p(j, i) = 1 - p(i, j)`, denn jedes Spiel hat denselben Ausgang, nur aus der Sicht des anderen. (Das spart die Hälfte
der Rechenzeit.) Das Paar `(i, i)` ist ein Spiel mit demselben Würfel und hat 50 Prozent.

Als Gesamtwert pro Würfel bilden wir den Mittelwert `pA` der Gewinnchancen gegen alle **anderen** Würfel. Wer den größten Mittelwert hat, ist der beste Würfel für jemanden, der den Würfel des Gegners nicht kennt.
`Platz` ordnet danach: Platz 1 bekommt der größte Wert.

In [ ]:
def tabelle(wuerfel, n):
    """Liefert (p, patt): p[i][j] = Gewinnchance von i gegen j, patt[i][j] = Anteil Patts."""
    k = len(wuerfel)
    p = [[0.5] * k for _ in range(k)]
    patt = [[0.0] * k for _ in range(k)]
    for i in range(k):
        for j in range(i + 1, k):
            siege, patts = vergleiche(wuerfel[i], wuerfel[j], n)
            p[i][j] = (siege + 0.5 * patts) / n
            p[j][i] = 1 - p[i][j]
            patt[i][j] = patts / n
            patt[j][i] = patts / n
    return p, patt


def mittel(p):
    k = len(p)
    return [sum(p[i][j] for j in range(k) if j != i) / (k - 1) for i in range(k)]


def zeige(p, wuerfel):
    k = len(p)
    pa = mittel(p)
    print('Spieler B:     ' + ' '.join('  W' + str(j + 1) + '  ' for j in range(k)) + '    pA  Platz')
    for i in range(k):
        platz = 1 + sum(1 for x in pa if x > pa[i] + 1e-12)
        zeile = ' '.join('%5.1f ' % (100 * p[i][j]) for j in range(k))
        print('Spieler A: W' + str(i + 1) + '  ' + zeile + '  %5.1f  %d' % (100 * pa[i], platz))
    for i in range(k):
        print('W' + str(i + 1) + ' =', wuerfel[i])

## Schritt 7: Alle Beispiele

Wir rechnen mit `n = 4000` Spielen je Paar. Die Zeilen sind Spieler A, die Spalten Spieler B. Die Zahlen sind Gewinnchancen von A in Prozent
(ein Spiel mit demselben Würfel auf beiden Seiten steht als 50,0).

#### wuerfel0.txt (Standardwürfel, `1 1 1 6 6 6`, 4er, 10er (0 bis 9), 12er und 20er)

In [ ]:
random.seed(40)
ERG = {}                                     # merkt sich die Tabellen für später
for nr in range(4):
    wuerfel = lies_wuerfel('beispieldaten/wuerfel' + str(nr) + '.txt')
    ERG[nr] = (wuerfel,) + tabelle(wuerfel, 4000)

zeige(ERG[0][1], ERG[0][0])

Spieler B:       W1     W2     W3     W4     W5     W6      pA  Platz
Spieler A: W1   50.0    1.3  100.0   64.5   65.0   88.8    63.9  2
Spieler A: W2   98.7   50.0  100.0   98.5   96.8   99.0    98.6  1
Spieler A: W3    0.0    0.0   50.0    0.0    0.0    0.0     0.0  6
Spieler A: W4   35.5    1.5  100.0   50.0   53.1   79.0    53.8  3
Spieler A: W5   35.0    3.2  100.0   46.9   50.0   75.2    52.1  4
Spieler A: W6   11.2    1.0  100.0   21.0   24.8   50.0    31.6  5
W1 = [1, 2, 3, 4, 5, 6]
W2 = [1, 1, 1, 6, 6, 6]
W3 = [1, 2, 3, 4]
W4 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
W5 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
W6 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


Der Würfel `1 1 1 6 6 6` (W2) ist mit Abstand der beste (98,6 Prozent im Mittel), danach kommt der normale Würfel (W1). Der 20-seitige Würfel (W6) von Anna landet auf Platz 5: Er würfelt nur mit einer
Wahrscheinlichkeit von 1/20 eine 6, und die Zielfelder verlangen oft genau passende kleine Zahlen. Clemens hat also mit seiner Überlegung zur 6 recht.

Der vierseitige Würfel W3 hat gar keine 6. Sein erster Stein läuft ins Ziel, ein zweiter kommt nie ins Spiel: Er verliert immer (0 Prozent), außer gegen sich selbst. Dann
steckt jeder irgendwann fest, das wäre ein Patt, und wir tragen 50,0 ein.

Wir stimmen mit der Tabelle der Lösungshinweise überein: dieselbe Reihenfolge der Plätze (W2, W1, W4, W5, W6, W3), und alle Einzelwerte weichen höchstens um 2 Prozentpunkte ab.

#### wuerfel1.txt (sechsseitige Würfel mit den Zahlen 1 bis 6, 2 bis 7, …, 6 bis 11)

In [ ]:
zeige(ERG[1][1], ERG[1][0])

Spieler B:       W1     W2     W3     W4     W5     W6      pA  Platz
Spieler A: W1   50.0   44.4   49.5   55.1   68.4   80.3    59.6  2
Spieler A: W2   55.6   50.0   51.4   53.4   65.2   74.4    60.0  1
Spieler A: W3   50.6   48.6   50.0   53.0   60.9   68.6    56.3  3
Spieler A: W4   44.9   46.6   47.0   50.0   60.2   65.6    52.8  4
Spieler A: W5   31.6   34.8   39.1   39.8   50.0   56.4    40.3  5
Spieler A: W6   19.7   25.6   31.4   34.4   43.6   50.0    30.9  6
W1 = [1, 2, 3, 4, 5, 6]
W2 = [2, 3, 4, 5, 6, 7]
W3 = [3, 4, 5, 6, 7, 8]
W4 = [4, 5, 6, 7, 8, 9]
W5 = [5, 6, 7, 8, 9, 10]
W6 = [6, 7, 8, 9, 10, 11]


Hier hat jeder Würfel sechs aufeinanderfolgende Zahlen, nur jeweils um 1 nach oben verschoben. Eine 6 hat jeder, aber höhere Würfel haben keine kleinen Zahlen mehr. Die Reihenfolge der Plätze ist
W2 (2 bis 7), W1 (1 bis 6), W3, W4, W5, W6 (6 bis 11), genau wie bei den Lösungshinweisen. W1 und W2 liegen mit 59,6 und 60,0 Prozent so eng beieinander, dass sie sich im Rahmen der Genauigkeit
(etwa ±1,6 Prozentpunkte) nicht unterscheiden. Größere Zahlen helfen also nicht, im Gegenteil: Der Würfel `6 bis 11` ist der schlechteste (30,9 Prozent).

#### wuerfel2.txt (Würfel mit 1 und 6, von „fünfmal 1, einmal 6“ bis „einmal 1, fünfmal 6“)

In [ ]:
zeige(ERG[2][1], ERG[2][0])

Spieler B:       W1     W2     W3     W4     W5      pA  Platz
Spieler A: W1   50.0    0.9    0.0    0.0    0.0     0.2  5
Spieler A: W2   99.1   50.0    3.7    0.1    0.0    25.7  4
Spieler A: W3  100.0   96.3   50.0    5.9    0.2    50.6  3
Spieler A: W4  100.0  100.0   94.2   50.0    7.6    75.4  2
Spieler A: W5  100.0  100.0   99.8   92.3   50.0    98.0  1
W1 = [1, 1, 1, 1, 1, 6]
W2 = [1, 1, 1, 1, 6, 6]
W3 = [1, 1, 1, 6, 6, 6]
W4 = [1, 1, 6, 6, 6, 6]
W5 = [1, 6, 6, 6, 6, 6]


Je mehr Seiten mit der 6, desto besser. Der Würfel mit fünf Sechsern (W5) gewinnt im Mittel 98,0 Prozent, der mit nur einer 6 (W1) fast nie (0,2 Prozent). Die Platzierung ist
streng nach der Anzahl der Sechser geordnet, genau wie in den Lösungshinweisen.

#### wuerfel3.txt (Würfel mit 4, 6, 8, 10, 12 und 20 Seiten)

In [ ]:
zeige(ERG[3][1], ERG[3][0])

Spieler B:       W1     W2     W3     W4     W5     W6      pA  Platz
Spieler A: W1   50.0   71.9   66.2   80.4   77.6   93.1    77.8  1
Spieler A: W2   28.1   50.0   47.8   64.5   64.7   88.8    58.8  3
Spieler A: W3   33.8   52.2   50.0   66.0   64.4   87.1    60.7  2
Spieler A: W4   19.6   35.5   34.0   50.0   51.1   79.5    43.9  4
Spieler A: W5   22.4   35.3   35.6   48.9   50.0   77.3    43.9  5
Spieler A: W6    6.9   11.3   12.9   20.5   22.7   50.0    14.8  6
W1 = [1, 2, 5, 6]
W2 = [1, 2, 3, 4, 5, 6]
W3 = [1, 2, 3, 4, 5, 6, 7, 8]
W4 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
W5 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
W6 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


Der Würfel `1 2 5 6` (W1) hat eine 6 in einem von vier Würfen und große Schritte. Er ist am besten (77,8 Prozent). Der 8-seitige Würfel (W3) liegt knapp vor dem normalen (W2). Plätze 4 und 5 (10er und 12er) liegen
mit je etwa 43,9 Prozent dicht beieinander (in den Lösungshinweisen 44,3 und 43,5). Der 20-seitige Würfel ist wieder der schlechteste. Die Plätze sind dieselben wie in den Lösungshinweisen: W1, W3, W2, W4, W5, W6.

## Patt: Wie oft kommt es vor?

Wir haben ein Patt als unentschieden gezählt. Wie oft das passiert, zeigt die folgende Tabelle für `wuerfel1.txt` (Anteil der Patts in Prozent aller Spiele des Paares).

In [ ]:
patt1 = ERG[1][2]
print('Patt in Prozent der Spiele (wuerfel1.txt):')
for i in range(6):
    print('W' + str(i + 1) + '  ' + ' '.join('%5.1f' % (100 * patt1[i][j]) for j in range(6)))

Patt in Prozent der Spiele (wuerfel1.txt):
W1    0.0   0.0   0.0   0.0   0.0   0.0
W2    0.0   0.0  11.5  12.9  17.0  20.8
W3    0.0  11.5   0.0  23.3  31.4  35.5
W4    0.0  12.9  23.3   0.0  38.6  43.7
W5    0.0  17.0  31.4  38.6   0.0  62.5
W6    0.0  20.8  35.5  43.7  62.5   0.0


Die Diagonale (derselbe Würfel gegen sich selbst) haben wir nicht simuliert, dort steht 0,0. In den Lösungshinweisen sind es zum Beispiel 6,2 Prozent für W2 gegen W2. Alle anderen Werte
entsprechen der Tabelle der Lösungshinweise (zum Beispiel W2 gegen W3: 11,5 gegenüber 10,7 und W5 gegen W6: 62,5 gegenüber 62,4).

Der Würfel `1 bis 6` (W1) steckt nie fest, die anderen immer häufiger, je größer ihre kleinste Zahl ist. Beim Würfel `6 bis 11` im Spiel gegen `5 bis 10` enden etwa 62 Prozent der Spiele im Patt.
Daher ist wichtig, das Patt zu behandeln. Würde man es zum Beispiel als Sieg für einen von beiden zählen, verschöbe sich die Tabelle erheblich. Die Hinweise zeigen, dass die Rangfolge gleich bleibt,
wenn man alle Patt-Spiele einfach wiederholt.

## Wie genau sind die Zahlen?

Ein einzelner Lauf ist Zufall. Wie stark die Schätzung schwankt, kann man selbst ausprobieren: Wir wiederholen den Vergleich `1 bis 6` gegen `2 bis 7` je zehnmal mit
verschiedener Spielzahl `n` und sehen, in welchem Bereich die Ergebnisse liegen.

In [ ]:
random.seed(7)
for n in [250, 1000, 4000]:
    anteile = []
    for wiederholung in range(10):
        siege, patts = vergleiche([1, 2, 3, 4, 5, 6], [2, 3, 4, 5, 6, 7], n)
        anteile.append(100 * (siege + 0.5 * patts) / n)
    print('n =', n, ': kleinster Wert', round(min(anteile), 1), ', größter Wert', round(max(anteile), 1))

n = 250 : kleinster Wert 38.0 , größter Wert 46.0
n = 1000 : kleinster Wert 41.6 , größter Wert 45.5
n = 4000 : kleinster Wert 42.5 , größter Wert 44.6


Bei `n = 250` liegen die zehn Ergebnisse noch um etwa 8 Prozentpunkte auseinander, bei `n = 4000` nur noch um etwa 2. Das passt zur Rechnung unten:
Die Streuung schrumpft mit der Wurzel aus `n`, für 4 mal so viele Spiele halbiert sie sich also nur.

## Für Neugierige: Wie viele Spiele sind genug?

Jedes Spiel ist wie ein Münzwurf, bei dem A mit der (unbekannten) Wahrscheinlichkeit `p` gewinnt. Die Schätzung `G / n` aus `n` Spielen streut mit der Standardabweichung
`wurzel(p (1 - p) / n)`. In 95 Prozent der Fälle liegt sie höchstens zwei Standardabweichungen daneben. Weil `p (1 - p)` höchstens `1/4` ist, gilt für die Abweichung
höchstens `1 / wurzel(n)`. Soll sie höchstens einen Prozentpunkt betragen (`0,01`), braucht man `n >= 10000`. Für `n = 4000` sind es etwa 1,6 Prozentpunkte. Die Lösungshinweise
rechnen mit `n = 120000` und kommen auf 0,3 Prozentpunkte.

**Weitere Spielvarianten**, die die Hinweise als Zusatz nennen und die man in `ziehe` einbauen könnte: Schlagzüge bevorzugen (auch mit hinteren Steinen), ohne Stein auf dem Startfeld beginnen,
nach einer 6 mehrmals würfeln dürfen oder auch zu dritt und zu viert spielen. Wichtig ist, immer **zuerst** die Regeln der Aufgabe zu simulieren und Varianten nur als Zusatz.